# CNIBP One-Click (VSCode + Colab)\n在 VSCode 的 Colab 插件连接远程 runtime 后，从上到下 `Run All`。

In [ ]:
# 只需要改这7项（默认：全量数据 + 先跑1个fold）
GIT_REPO = 'https://github.com/67vmg9wrfn-beep/Lab.git'
GIT_BRANCH = 'main'
PROJECT_SUBDIR = '06_experiments/cnibp/repro_ppg_bp'
RUN_COMPARE_TWO_MODES = False  # 默认单配置训练
SMOKE_MODE = False  # 默认全量数据
RUN_CONFIG_PATH = 'configs/paper_repro_fold1_check.json'  # 默认先跑1个fold验证
FORCE_NO_REUSE_RESUME = True  # True: 本次强制不复用/不断点续训，避免串到旧实验



In [ ]:
import os, shutil
from google.colab import drive

if os.path.ismount('/content/drive'):
    get_ipython().system('fusermount -u /content/drive')
if os.path.exists('/content/drive'):
    shutil.rmtree('/content/drive')

os.makedirs('/content/drive', exist_ok=True)
drive.mount('/content/drive', force_remount=True)


In [ ]:
import os, shutil, subprocess
if os.path.exists('/content/repo_src'):
    shutil.rmtree('/content/repo_src')
subprocess.run(['git','clone','--depth','1','--branch', GIT_BRANCH, GIT_REPO, '/content/repo_src'], check=True)
print('git clone done')


In [ ]:
import os
PROJECT_ROOT = f'/content/repo_src/{PROJECT_SUBDIR}'
print('PROJECT_ROOT=', PROJECT_ROOT)
assert os.path.exists(PROJECT_ROOT), f'Path not found: {PROJECT_ROOT}'

In [ ]:
from pathlib import Path
import re

BASE = Path('/content/drive/MyDrive')
if not BASE.exists():
    raise FileNotFoundError('Drive not mounted at /content/drive/MyDrive')

mat_files = [f for f in BASE.rglob('*.mat') if '/__MACOSX/' not in str(f)]
print(f'[INFO] total .mat files found (without __MACOSX): {len(mat_files)}')

pat = re.compile(r'^part_([0-9]+)\.mat$', re.IGNORECASE)
by_dir = {}
for f in mat_files:
    m = pat.match(f.name)
    if not m:
        continue
    d = str(f.parent)
    by_dir.setdefault(d, {})[int(m.group(1))] = str(f)

if not by_dir:
    raise FileNotFoundError('No Part_*.mat files found under /content/drive/MyDrive')

dirs = sorted(by_dir.keys(), key=lambda d: (len(by_dir[d]), 'kachuee' in d.lower() and 'raw_mat' in d.lower()), reverse=True)
DATA_ROOT = dirs[0]
part_map = by_dir[DATA_ROOT]
part_ids = sorted(part_map.keys())
PARTS = [part_map[i] for i in part_ids]  # absolute paths

if SMOKE_MODE:
    if 1 in part_map:
        PARTS = [part_map[1]]
    else:
        PARTS = [PARTS[0]]

print('[OK] DATA_ROOT =', DATA_ROOT)
print('[OK] detected parts (absolute) =', PARTS)
print('[OK] SMOKE_MODE =', SMOKE_MODE)
if 0 not in part_ids:
    print('[WARN] Part_0.mat not found; running with available parts only.')
if not SMOKE_MODE and len(PARTS) < 2:
    raise RuntimeError('Too few parts to train reliably. Need at least 2 Part_*.mat files.')



In [ ]:
import subprocess
subprocess.run(['python','-m','pip','install','-r', f'{PROJECT_ROOT}/requirements_colab.txt'], check=True)
subprocess.run(['python','-m','pip','install','-e', PROJECT_ROOT], check=True)
print('dependencies installed')

import subprocess
subprocess.run(['git','-C','/content/repo_src','rev-parse','--short','HEAD'], check=False)
print('[NOTE] expect commit >= a23e8db')



In [ ]:
import os, glob, subprocess, json
OUT_ROOT='/content/drive/MyDrive/cnibp_repro_outputs'
os.makedirs(OUT_ROOT, exist_ok=True)
parts_arg=','.join(PARTS)

cfg_single = f'{PROJECT_ROOT}/configs/paper_repro_smoke.json' if SMOKE_MODE else f'{PROJECT_ROOT}/configs/paper_repro.json'
cfg_a = f'{PROJECT_ROOT}/configs/paper_repro_smoke.json' if SMOKE_MODE else f'{PROJECT_ROOT}/configs/paper_repro.json'
cfg_b = f'{PROJECT_ROOT}/configs/paper_plus_segmentz_abp_robust_smoke.json' if SMOKE_MODE else f'{PROJECT_ROOT}/configs/paper_plus_segmentz_abp_robust.json'
if RUN_CONFIG_PATH:
    cfg_single = f"{PROJECT_ROOT}/{RUN_CONFIG_PATH}"

def _read_json(path):
    with open(path, 'r', encoding='utf-8') as f:
        return json.load(f)

def _pre_sig(cfg):
    keys = [
        'fs','window_sec','overlap','min_duration_sec','abp_max','flatline_std_threshold',
        'ppg_norm_mode','zscore_eps','abp_label_mode','abp_filter_mode','abp_lowpass_hz',
        'abp_peak_distance_sec','abp_peak_prominence','sbp_min','sbp_max_label','dbp_min',
        'dbp_max_label','pulse_pressure_min','pulse_pressure_max'
    ]
    return {k: cfg.get(k) for k in keys}

def _train_sig(cfg):
    keys = [
        'seed','n_splits','batch_size','max_epochs','patience','lr','subject_level_split',
        'num_workers','pin_memory','persistent_workers','prefetch_factor','use_amp'
    ]
    return {k: cfg.get(k) for k in keys}

def _run_cfg(run_dir):
    p = f'{run_dir}/resolved_config.json'
    if not os.path.exists(p):
        return None
    try:
        return _read_json(p)
    except Exception:
        return None

def _has_pre_artifacts(run_dir):
    pre = f'{run_dir}/preprocess'
    req = ['preprocessed_manifest.json','segments_meta.csv','X.float16.mmap','y.float32.mmap']
    return all(os.path.exists(f'{pre}/{x}') for x in req)

def _find_latest_compatible_run(target_cfg, need_train=False):
    for d in sorted(glob.glob(f'{OUT_ROOT}/run_*'), reverse=True):
        rc = _run_cfg(d)
        if rc is None:
            continue
        if _pre_sig(rc) != _pre_sig(target_cfg):
            continue
        if need_train:
            if not os.path.isdir(f'{d}/train'):
                continue
            if _train_sig(rc) != _train_sig(target_cfg):
                continue
            return d
        else:
            if _has_pre_artifacts(d):
                return d
    return ''

if RUN_COMPARE_TWO_MODES:
    log_file=f'{OUT_ROOT}/last_compare.log'
    cmd=[
        'python','-m','cnibp_repro.run_compare',
        '--drive_root', DATA_ROOT,
        '--parts', parts_arg,
        '--output_root', OUT_ROOT,
        '--config_a', cfg_a,
        '--config_b', cfg_b,
        '--label_a', 'paper_raw_smoke' if SMOKE_MODE else 'paper_raw',
        '--label_b', 'segmentz_abp_robust_smoke' if SMOKE_MODE else 'segmentz_abp_robust'
    ]
    print('[INFO] compare modes = paper_raw vs segmentz_abp_robust')
    print('[INFO] smoke mode =', SMOKE_MODE)
else:
    log_file=f'{OUT_ROOT}/last_run.log'
    target_cfg = _read_json(cfg_single)

    reuse_from = _find_latest_compatible_run(target_cfg, need_train=False)
    resume_from = _find_latest_compatible_run(target_cfg, need_train=True)

    if FORCE_NO_REUSE_RESUME:
        reuse_from = ''
        resume_from = ''

    cmd=[
        'python','-m','cnibp_repro.run_repro',
        '--drive_root', DATA_ROOT,
        '--parts', parts_arg,
        '--config', cfg_single,
        '--output_root',OUT_ROOT
    ]
    if reuse_from:
        cmd += ['--reuse_from_run_dir', reuse_from]
    if resume_from:
        cmd += ['--resume_from_run_dir', resume_from]

    print('[INFO] run parts (absolute) =', parts_arg)
    print('[INFO] force_no_reuse_resume =', FORCE_NO_REUSE_RESUME)
    print('[INFO] config =', cfg_single)
    print('[INFO] smoke mode =', SMOKE_MODE)
    print('[INFO] reuse_from_run_dir =', reuse_from if reuse_from else '(none)')
    print('[INFO] resume_from_run_dir =', resume_from if resume_from else '(none)')

with open(log_file, 'w', encoding='utf-8') as f:
    proc=subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in proc.stdout:
        print(line, end='')
        f.write(line)
    code=proc.wait()
if code != 0:
    raise RuntimeError(f'run failed, see {log_file}')
print(f'log saved: {log_file}')
if RUN_COMPARE_TWO_MODES:
    print(f'compare reports under: {OUT_ROOT}/compare_runs')




In [ ]:
# 失败时优先看这里
print('/content/drive/MyDrive/cnibp_repro_outputs/last_run.log')
print('/content/drive/MyDrive/cnibp_repro_outputs/last_compare.log')
print('/content/drive/MyDrive/cnibp_repro_outputs/compare_runs')

